In [0]:
%sql

CREATE OR REPLACE TABLE dev.claims_project.gold_suspicious_claims AS
    WITH PROVIDER_MONTHLY AS(
        SELECT * FROM dev.claims_project.gold_provider_monthly
    )
        ,
    
    LOCATION_BASELINE AS (
        SELECT
            PROVIDERLOCATION,
            SERVICE_MONTH,
            AVG(TOTAL_CLAIMS) AS avg_claims,
            STDDEV(TOTAL_CLAIMS) AS STDDEV_CLAIMS,
            AVG(TOTAL_PAID) AS avg_total_paid,
            STDDEV(TOTAL_PAID) AS stddev_total_paid,
            AVG(avg_claim_amount) AS avg_claim_amount_baseline,
            STDDEV(avg_claim_amount) AS stddev_claim_amount
        FROM PROVIDER_MONTHLY GROUP BY PROVIDERLOCATION, SERVICE_MONTH
    ),

    SAME_DAY_CLAIMS AS (
        SELECT 
            PROVIDERID,
            PATIENTID,
            CLAIMDATE,
            COUNT(DISTINCT CLAIMID) AS SAME_DAY_CLAIM_COUNT,
            ROUND(SUM(CLAIMAMOUNT), 2) AS SAME_DAY_TOTAL_PAID
        FROM dev.claims_project.silver_claims_enriched
        GROUP BY PROVIDERID, PATIENTID, CLAIMDATE
        HAVING COUNT(DISTINCT CLAIMID) >= 3

    ),

    PROVIDER_FLAGS AS (
        SELECT
            P.PROVIDERID,
            P.PROVIDERLOCATION,
            P.PROVIDERSPECIALTY,
            P.SERVICE_MONTH,
            P.TOTAL_CLAIMS,
            P.TOTAL_PATIENTS,
            P.TOTAL_PAID,
            P.AVG_CLAIM_AMOUNT,
            P.AVG_CLAIM_LAG_DAYS,
            P.PREVIOUS_MONTH_PAID,
            P.PAID_CHANGE_FROM_PREVIOUS_MONTH,
            P.PAID_PERCENT_CHANGE,
            P.ROLLING_3_MONTH_AVG_PAID,
            B.AVG_CLAIMS,
            B.STDDEV_CLAIMS,
            B.AVG_TOTAL_PAID,
            B.STDDEV_TOTAL_PAID,
            B.AVG_CLAIM_AMOUNT_BASELINE,
            B.STDDEV_CLAIM_AMOUNT,

            CASE
                WHEN P.TOTAL_CLAIMS > B.AVG_CLAIMS + 2 * B.STDDEV_CLAIMS
                THEN 1
                ELSE 0
            END AS HIGH_CLAIM_VOLUME_FLAG,

            CASE 
                WHEN P.TOTAL_PAID > B.AVG_TOTAL_PAID + 2 * B.STDDEV_TOTAL_PAID
                THEN 1
                ELSE 0
            END AS HIGH_PAID_AMOUNT_FLAG,

            CASE 
                WHEN P.PAID_PERCENT_CHANGE >= 100
                THEN 1
                ELSE 0
            END AS SUDDEN_PAID_SPIKE_FLAG,

            CASE 
                WHEN P.AVG_CLAIM_LAG_DAYS > 30
                THEN 1
                ELSE 0
            END AS DELAYED_BILLING_FLAG,

            CASE
                WHEN P.AVG_CLAIM_AMOUNT > B.AVG_CLAIM_AMOUNT_BASELINE + 2 * B.STDDEV_CLAIM_AMOUNT
                THEN 1
                ELSE 0
            END AS ABNORMAL_AVG_CLAIM_FLAG

        FROM PROVIDER_MONTHLY P

        LEFT JOIN LOCATION_BASELINE B
        ON P.PROVIDERLOCATION = B.PROVIDERLOCATION
        AND P.SERVICE_MONTH = B.SERVICE_MONTH
    )


    SELECT
    ProviderID,
    providerspecialty,
    providerlocation,
    service_month,
    total_claims,
    total_patients,
    total_paid,
    avg_claim_amount,
    avg_claim_lag_days,
    paid_percent_change,
    rolling_3_month_avg_paid,
    high_claim_volume_flag,
    high_paid_amount_flag,
    sudden_paid_spike_flag,
    delayed_billing_flag,
    abnormal_avg_claim_flag,
    (high_claim_volume_flag + high_paid_amount_flag + sudden_paid_spike_flag + delayed_billing_flag + abnormal_avg_claim_flag) AS suspicion_score,
    CASE
        WHEN (high_claim_volume_flag + high_paid_amount_flag + sudden_paid_spike_flag + delayed_billing_flag + abnormal_avg_claim_flag) >= 4
        THEN 'HIGH'

        WHEN (high_claim_volume_flag + high_paid_amount_flag + sudden_paid_spike_flag + delayed_billing_flag+ abnormal_avg_claim_flag) >= 2
        THEN 'MEDIUM'

        WHEN (high_claim_volume_flag + high_paid_amount_flag + sudden_paid_spike_flag + delayed_billing_flag+ abnormal_avg_claim_flag)= 1
        THEN 'LOW'

        ELSE 'NORMAL'

    END AS suspicion_level

FROM provider_flags;
